In [1]:
import geopandas as gpd


In [2]:
buildings = gpd.read_file("../data/processed/buildings_with_exposure.gpkg")


In [3]:
flood = gpd.read_file("../data/processed/flood_plains_fixed.gpkg")
flood = flood[flood.intersects(buildings.union_all().envelope)]   # keep only polygons near the study area

In [13]:
critical = buildings[(buildings["exposed"] == 1) & (buildings["use"] != "Unknown")]

# Keep only the columns the map needs; date columns can break kepler's export
cols = ["building_id", "name", "use", "suburb_locality", "exposed", "geometry"]
buildings_map = buildings[cols].to_crs(4326)
buildings_map["exposure"] = buildings_map["exposed"].map({0: "Not exposed", 1: "Exposed"})
critical_map = critical[cols].to_crs(4326)
flood_map = flood[["geometry"]].to_crs(4326)

print(len(buildings_map), len(critical_map), len(flood_map))

22772 44 239


In [21]:
from keplergl import KeplerGl

m = KeplerGl(height=600)

m.add_data(data=flood_map, name="Flood plain")
m.add_data(data=buildings_map, name="Buildings")
m.add_data(data=critical_map, name="Critical facilities")
m

In [24]:
import json

with open("../outputs/map_config.json", "w") as f:
    json.dump(m.config, f)

m.save_to_html(file_name="../outputs/flood_map.html")

Map saved to ../outputs/flood_map.html!
